In [ ]:
from pathlib import Path

import json
import random
import time
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

from tensorflow.keras.preprocessing.image import ImageDataGenerator


SEED = 12345
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
NUM_CLASSES = 35

RUN_NAME = "custom_cnn_clean_split_lanczos_oversampling_to100_seed12345"

OUTPUT_DIR = Path("/kaggle/working") / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)


print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

print("\nRun:", RUN_NAME)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Classes:", NUM_CLASSES)
print("Output directory:", OUTPUT_DIR)

In [ ]:
KAGGLE_INPUT = Path("/kaggle/input")


MANIFEST_PATH = next(
    KAGGLE_INPUT.rglob("zoolake_clean_split_manifest.csv")
)

CLASS_NAMES_PATH = next(
    KAGGLE_INPUT.rglob("zoolake_class_names.json")
)

DATASET_ROOT = next(
    path
    for path in KAGGLE_INPUT.rglob("zooplankton_0p5x")
    if path.is_dir()
)


manifest = pd.read_csv(MANIFEST_PATH)

with open(CLASS_NAMES_PATH, "r") as file:
    CLASS_NAMES = json.load(file)


manifest["filepath"] = manifest["relative_filepath"].apply(
    lambda path: str(DATASET_ROOT / path)
)


train_df = (
    manifest[manifest["split"] == "train"]
    .copy()
    .reset_index(drop=True)
)

val_df = (
    manifest[manifest["split"] == "validation"]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    manifest[manifest["split"] == "test"]
    .copy()
    .reset_index(drop=True)
)


print("Manifest:", MANIFEST_PATH)
print("Class names:", CLASS_NAMES_PATH)
print("Dataset root:", DATASET_ROOT)

print("\nTraining:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Classes:", len(CLASS_NAMES))

In [ ]:
RARE_CLASS_THRESHOLD = 50
OVERSAMPLING_TARGET = 100


train_class_counts_before = (
    train_df["label"]
    .value_counts()
    .reindex(CLASS_NAMES, fill_value=0)
    .astype(int)
)


rare_classes = [
    label
    for label in CLASS_NAMES
    if train_class_counts_before[label]
    < RARE_CLASS_THRESHOLD
]


if len(rare_classes) != 6:
    raise ValueError(
        "Expected exactly 6 training classes with fewer "
        f"than {RARE_CLASS_THRESHOLD} images, but found "
        f"{len(rare_classes)}: {rare_classes}"
    )


oversampling_parts = [train_df.copy()]


for class_offset, label in enumerate(rare_classes):
    class_df = train_df[
        train_df["label"] == label
    ]

    samples_to_add = (
        OVERSAMPLING_TARGET - len(class_df)
    )

    if samples_to_add <= 0:
        continue

    sampled_rows = class_df.sample(
        n=samples_to_add,
        replace=True,
        random_state=SEED + class_offset + 1,
    )

    oversampling_parts.append(sampled_rows)


oversampled_train_df = (
    pd.concat(
        oversampling_parts,
        ignore_index=True,
    )
    .sample(
        frac=1.0,
        random_state=SEED,
    )
    .reset_index(drop=True)
)


train_class_counts_after = (
    oversampled_train_df["label"]
    .value_counts()
    .reindex(CLASS_NAMES, fill_value=0)
    .astype(int)
)


for label in CLASS_NAMES:
    if label in rare_classes:
        expected_count = OVERSAMPLING_TARGET
    else:
        expected_count = train_class_counts_before[label]

    if train_class_counts_after[label] != expected_count:
        raise ValueError(
            f"Unexpected oversampled count for {label}: "
            f"{train_class_counts_after[label]} instead of "
            f"{expected_count}."
        )


oversampling_summary_df = pd.DataFrame({
    "class": CLASS_NAMES,
    "original_train_count": [
        int(train_class_counts_before[label])
        for label in CLASS_NAMES
    ],
    "oversampled_train_count": [
        int(train_class_counts_after[label])
        for label in CLASS_NAMES
    ],
})

oversampling_summary_df["added_rows"] = (
    oversampling_summary_df[
        "oversampled_train_count"
    ]
    - oversampling_summary_df[
        "original_train_count"
    ]
)

oversampling_summary_df["oversampled"] = (
    oversampling_summary_df["class"]
    .isin(rare_classes)
)


OVERSAMPLING_SUMMARY_PATH = (
    OUTPUT_DIR
    / "train_class_distribution_before_after.csv"
)

oversampling_summary_df.to_csv(
    OVERSAMPLING_SUMMARY_PATH,
    index=False,
)


print("Rare-class threshold:", RARE_CLASS_THRESHOLD)
print("Oversampling target:", OVERSAMPLING_TARGET)
print("Rare classes:", rare_classes)

print("\nOriginal training rows:", len(train_df))
print(
    "Oversampled training rows:",
    len(oversampled_train_df),
)
print(
    "Added training rows:",
    len(oversampled_train_df) - len(train_df),
)

print("\nRare-class counts before and after:")
print(
    oversampling_summary_df[
        oversampling_summary_df["oversampled"]
    ].to_string(index=False)
)

print(
    "\nSaved oversampling summary:",
    OVERSAMPLING_SUMMARY_PATH,
)


In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=180,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.20,
    shear_range=10,
)

eval_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
)


train_generator = train_datagen.flow_from_dataframe(
    dataframe=oversampled_train_df,
    x_col="filepath",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=SEED,
    interpolation="lanczos",
)

val_generator = eval_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col="filepath",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
    interpolation="lanczos",
)


print("\nTraining batches:", len(train_generator))
print("Validation batches:", len(val_generator))

In [ ]:
tf.keras.backend.clear_session()


model = tf.keras.Sequential([
    tf.keras.layers.Input(
        shape=(*IMAGE_SIZE, 3)
    ),

    tf.keras.layers.Conv2D(
        32,
        kernel_size=3,
        padding="same",
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(
        64,
        kernel_size=3,
        padding="same",
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(
        128,
        kernel_size=3,
        padding="same",
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(
        128,
        kernel_size=3,
        padding="same",
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(
        128,
        activation="relu",
    ),

    tf.keras.layers.Dropout(
        0.5
    ),

    tf.keras.layers.Dense(
        NUM_CLASSES,
        activation="softmax",
    ),
])


model.summary()

In [ ]:
LEARNING_RATE = 1e-4
MAX_EPOCHS = 200
EARLY_STOPPING_PATIENCE = 30


BEST_MODEL_PATH = OUTPUT_DIR / "best_custom_cnn.keras"
HISTORY_PATH = OUTPUT_DIR / "training_history.csv"


model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE,
    ),
    loss="categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.TopKCategoricalAccuracy(
            k=2,
            name="top2_accuracy",
        ),
    ],
)


callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1,
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=1,
    ),

    tf.keras.callbacks.CSVLogger(
        str(HISTORY_PATH)
    ),
]


print("Learning rate:", LEARNING_RATE)
print("Maximum epochs:", MAX_EPOCHS)
print("Early-stopping patience:", EARLY_STOPPING_PATIENCE)
print("Total parameters:", model.count_params())
print("Best model:", BEST_MODEL_PATH)
print("History:", HISTORY_PATH)

In [ ]:
train_generator.reset()
val_generator.reset()


training_started = time.time()


history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=2,
)


training_seconds = time.time() - training_started
training_hours = training_seconds / 3600


history_df = pd.DataFrame(history.history)

history_df.insert(
    0,
    "epoch",
    np.arange(1, len(history_df) + 1),
)

history_df.to_csv(
    HISTORY_PATH,
    index=False,
)


best_epoch = int(
    history_df.loc[
        history_df["val_loss"].idxmin(),
        "epoch",
    ]
)

best_val_loss = float(
    history_df["val_loss"].min()
)


training_info = {
    "epochs_completed": len(history_df),
    "best_epoch": best_epoch,
    "best_validation_loss": best_val_loss,
    "training_seconds": training_seconds,
    "training_hours": training_hours,
}

with open(
    OUTPUT_DIR / "training_info.json",
    "w",
) as file:
    json.dump(
        training_info,
        file,
        indent=2,
    )


print("\nTraining completed.")
print("Epochs completed:", len(history_df))
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)
print("Training hours:", training_hours)

In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5),
)


axes[0].plot(
    history_df["epoch"],
    history_df["accuracy"],
    label="Training",
)

axes[0].plot(
    history_df["epoch"],
    history_df["val_accuracy"],
    label="Validation",
)

axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].grid(alpha=0.3)
axes[0].legend()


axes[1].plot(
    history_df["epoch"],
    history_df["loss"],
    label="Training",
)

axes[1].plot(
    history_df["epoch"],
    history_df["val_loss"],
    label="Validation",
)

axes[1].axvline(
    best_epoch,
    color="red",
    linestyle="--",
    label=f"Best epoch: {best_epoch}",
)

axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].grid(alpha=0.3)
axes[1].legend()


axes[2].plot(
    history_df["epoch"],
    history_df["top2_accuracy"],
    label="Training",
)

axes[2].plot(
    history_df["epoch"],
    history_df["val_top2_accuracy"],
    label="Validation",
)

axes[2].set_title("Top-2 Accuracy")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Top-2 Accuracy")
axes[2].grid(alpha=0.3)
axes[2].legend()


plt.suptitle(
    "Custom CNN — Lanczos",
    fontsize=15,
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "training_curves.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close()

In [ ]:
best_model = tf.keras.models.load_model(
    BEST_MODEL_PATH,
    compile=False,
)


test_generator = eval_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="filepath",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
    interpolation="lanczos",
)


test_generator.reset()

y_prob = best_model.predict(
    test_generator,
    verbose=1,
)


y_true = np.asarray(
    test_generator.classes,
    dtype=np.int64,
)

y_pred = np.argmax(
    y_prob,
    axis=1,
)


test_loss = float(
    -np.mean(
        np.log(
            np.clip(
                y_prob[
                    np.arange(len(y_true)),
                    y_true,
                ],
                1e-7,
                1.0,
            )
        )
    )
)

test_accuracy = float(
    accuracy_score(
        y_true,
        y_pred,
    )
)

macro_precision = float(
    precision_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
)

macro_recall = float(
    recall_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
)

macro_f1 = float(
    f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
)

top2_predictions = np.argsort(
    y_prob,
    axis=1,
)[:, -2:]

top2_accuracy = float(
    np.mean(
        np.any(
            top2_predictions == y_true[:, None],
            axis=1,
        )
    )
)


metrics = {
    "model": "Custom CNN",
    "split": "test",
    "samples": len(y_true),
    "loss": test_loss,
    "accuracy": test_accuracy,
    "macro_precision": macro_precision,
    "macro_recall": macro_recall,
    "macro_f1": macro_f1,
    "top2_accuracy": top2_accuracy,
    "best_epoch": best_epoch,
    "seed": SEED,
}


with open(
    OUTPUT_DIR / "test_metrics.json",
    "w",
) as file:
    json.dump(
        metrics,
        file,
        indent=2,
    )


print("\nTEST RESULTS")

for name, value in metrics.items():
    print(f"{name}: {value}")

In [ ]:
classification_results = classification_report(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)


classification_df = (
    pd.DataFrame(classification_results)
    .transpose()
)

classification_df.to_csv(
    OUTPUT_DIR / "classification_report.csv"
)


predictions_df = pd.DataFrame({
    "image_name": [
        Path(path).name
        for path in test_generator.filepaths
    ],
    "true_class": [
        CLASS_NAMES[index]
        for index in y_true
    ],
    "predicted_class": [
        CLASS_NAMES[index]
        for index in y_pred
    ],
    "confidence": np.max(
        y_prob,
        axis=1,
    ),
    "correct": y_true == y_pred,
})


predictions_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False,
)


np.savez_compressed(
    OUTPUT_DIR / "predictions.npz",
    y_true=y_true,
    y_pred=y_pred,
    y_prob=y_prob.astype(np.float32),
    class_names=np.asarray(CLASS_NAMES),
    image_names=np.asarray(
        [
            Path(path).name
            for path in test_generator.filepaths
        ]
    ),
)


print("Classification report saved.")
print("Test predictions saved.")
print("Prediction arrays saved.")

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES),
)


cm_normalized = (
    cm.astype(float)
    / np.maximum(
        cm.sum(axis=1, keepdims=True),
        1,
    )
)


support = cm.sum(axis=1)

sort_order = np.argsort(
    support
)[::-1]


cm_sorted = cm[
    sort_order
][:, sort_order]

cm_normalized_sorted = cm_normalized[
    sort_order
][:, sort_order]


class_names_sorted = [
    CLASS_NAMES[index]
    for index in sort_order
]

support_sorted = support[
    sort_order
]


class_labels_with_support = [
    f"{name} ({int(class_support)})"
    for name, class_support in zip(
        class_names_sorted,
        support_sorted,
    )
]


pd.DataFrame(
    cm_sorted,
    index=class_labels_with_support,
    columns=class_names_sorted,
).to_csv(
    OUTPUT_DIR / "confusion_matrix_counts.csv"
)


pd.DataFrame(
    cm_normalized_sorted,
    index=class_labels_with_support,
    columns=class_names_sorted,
).to_csv(
    OUTPUT_DIR / "confusion_matrix_normalized.csv"
)


figure_size = max(
    12,
    0.38 * NUM_CLASSES,
)

plt.figure(
    figsize=(
        figure_size,
        figure_size,
    )
)


image = plt.imshow(
    cm_normalized_sorted,
    interpolation="nearest",
    cmap="Blues",
    vmin=0,
    vmax=1,
)


plt.colorbar(
    image,
    fraction=0.046,
    pad=0.04,
    label="Normalized count",
)


tick_positions = np.arange(
    NUM_CLASSES
)


plt.xticks(
    tick_positions,
    class_names_sorted,
    rotation=90,
    fontsize=7,
)

plt.yticks(
    tick_positions,
    class_labels_with_support,
    fontsize=7,
)


plt.title(
    "Custom CNN - Normalized Confusion Matrix"
)

plt.xlabel(
    "Predicted class"
)

plt.ylabel(
    "True class"
)


for row in range(NUM_CLASSES):
    for column in range(NUM_CLASSES):

        value = cm_normalized_sorted[
            row,
            column,
        ]

        if row == column or value >= 0.05:

            text_color = (
                "white"
                if value >= 0.50
                else "black"
            )

            plt.text(
                column,
                row,
                f"{value:.2f}",
                ha="center",
                va="center",
                color=text_color,
                fontsize=5,
            )


plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "confusion_matrix_normalized.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
per_class_df = (
    classification_df
    .loc[CLASS_NAMES]
    .reset_index()
    .rename(
        columns={
            "index": "class",
            "f1-score": "f1_score",
        }
    )
)


per_class_df = per_class_df.sort_values(
    "support",
    ascending=False,
)


per_class_df.to_csv(
    OUTPUT_DIR / "per_class_metrics.csv",
    index=False,
)


per_class_df["class_with_support"] = (
    per_class_df.apply(
        lambda row: (
            f"{row['class']} "
            f"({int(row['support'])})"
        ),
        axis=1,
    )
)


heatmap_data = per_class_df.set_index(
    "class_with_support"
)[
    [
        "precision",
        "recall",
        "f1_score",
    ]
]


figure_height = max(
    8,
    0.35 * len(heatmap_data),
)


plt.figure(
    figsize=(
        8,
        figure_height,
    )
)


image = plt.imshow(
    heatmap_data.values,
    aspect="auto",
    vmin=0,
    vmax=1,
    cmap="RdBu",
)


plt.colorbar(
    image,
    label="Score",
)


plt.xticks(
    np.arange(3),
    [
        "Precision",
        "Recall",
        "F1-score",
    ],
)


plt.yticks(
    np.arange(
        len(heatmap_data)
    ),
    heatmap_data.index,
)


plt.title(
    "Custom CNN - Per-Class Metrics"
)

plt.xlabel(
    "Metric"
)

plt.ylabel(
    "Class"
)


for row in range(
    len(heatmap_data)
):
    for column in range(3):

        value = heatmap_data.values[
            row,
            column,
        ]

        text_color = (
            "white"
            if value >= 0.75
            else "black"
        )

        plt.text(
            column,
            row,
            f"{value:.2f}",
            ha="center",
            va="center",
            color=text_color,
            fontsize=8,
        )


plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "per_class_metrics.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
run_config = {
    "run_name": RUN_NAME,
    "model": "Custom CNN",
    "training_type": "from scratch",
    "seed": SEED,
    "image_size": list(IMAGE_SIZE),
    "batch_size": BATCH_SIZE,
    "classes": NUM_CLASSES,
    "train_samples": len(oversampled_train_df),
    "train_samples_original": len(train_df),
    "train_samples_after_oversampling": len(oversampled_train_df),
    "validation_samples": len(val_df),
    "test_samples": len(test_df),
    "learning_rate": LEARNING_RATE,
    "maximum_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "checkpoint_monitor": "val_loss",
    "loss": "categorical_crossentropy",
    "sampling": "random_oversampling",
    "interpolation": "lanczos",
    "oversampling": {
        "selection_rule": "original_train_count < 50",
        "rare_class_threshold": RARE_CLASS_THRESHOLD,
        "target_count_per_rare_class": OVERSAMPLING_TARGET,
        "sampling_with_replacement": True,
        "rare_classes": rare_classes,
        "added_training_rows": (
            len(oversampled_train_df) - len(train_df)
        ),
        "validation_unchanged": True,
        "test_unchanged": True,
    },
    "class_weights": None,
    "augmentation": {
        "rotation_range": 180,
        "horizontal_flip": True,
        "vertical_flip": True,
        "zoom_range": 0.20,
        "shear_range": 10,
    },
}


with open(
    OUTPUT_DIR / "run_config.json",
    "w",
) as file:
    json.dump(
        run_config,
        file,
        indent=2,
    )


pd.DataFrame(
    [metrics]
).to_csv(
    OUTPUT_DIR / "summary_metrics.csv",
    index=False,
)


with open(
    OUTPUT_DIR / "model_summary.txt",
    "w",
) as file:

    best_model.summary(
        print_fn=lambda line: file.write(
            line + "\n"
        )
    )


ARCHIVE_PATH = (
    Path("/kaggle/working")
    / f"{RUN_NAME}_results.zip"
)


with zipfile.ZipFile(
    ARCHIVE_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    for path in sorted(
        OUTPUT_DIR.rglob("*")
    ):
        if path.is_file():

            archive.write(
                path,
                arcname=path.relative_to(
                    OUTPUT_DIR
                ),
            )


print("Final results archive:")
print(ARCHIVE_PATH)

print("\nFiles included:")

for path in sorted(
    OUTPUT_DIR.rglob("*")
):
    if path.is_file():
        print(" -", path.name)